In [1]:
"""
Backpropagation — Multi-Layer Perceptron implemented from scratch
Architecture: Input → Hidden (sigmoid) → Output (sigmoid)
Loss: Binary Cross-Entropy
"""

import math
import random


# ── Activation & Loss ─────────────────────────────────────────────────────────

def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-9  # prevent log(0)
    return -sum(
        t * math.log(p + eps) + (1 - t) * math.log(1 - p + eps)
        for t, p in zip(y_true, y_pred)
    ) / len(y_true)


# ── Network ───────────────────────────────────────────────────────────────────

class MLP:
    """Simple 3-layer MLP trained with backpropagation."""

    def __init__(self, n_input, n_hidden, n_output, lr=0.1, seed=42):
        random.seed(seed)
        self.lr = lr

        # Xavier initialisation
        def rand_w(fan_in, fan_out):
            limit = math.sqrt(6 / (fan_in + fan_out))
            return [[random.uniform(-limit, limit) for _ in range(fan_in)]
                    for _ in range(fan_out)]

        def rand_b(size):
            return [0.0] * size

        # Weights & biases: input→hidden
        self.W1 = rand_w(n_input, n_hidden)
        self.b1 = rand_b(n_hidden)

        # Weights & biases: hidden→output
        self.W2 = rand_w(n_hidden, n_output)
        self.b2 = rand_b(n_output)

    # ── helpers ──

    def _dot(self, weights_row, inputs):
        return sum(w * x for w, x in zip(weights_row, inputs))

    def _layer(self, W, b, inputs):
        """Compute pre-activations (z) and activations (a) for one layer."""
        z = [self._dot(W[j], inputs) + b[j] for j in range(len(b))]
        a = [sigmoid(zj) for zj in z]
        return z, a

    # ── forward pass ──

    def forward(self, x):
        self.x = x
        self.z1, self.a1 = self._layer(self.W1, self.b1, x)
        self.z2, self.a2 = self._layer(self.W2, self.b2, self.a1)
        return self.a2

    # ── backward pass ──

    def backward(self, y_true):
        n_out = len(self.b2)
        n_hid = len(self.b1)

        # Output layer gradient  (∂L/∂a2 · sigmoid')
        delta2 = [
            (self.a2[k] - y_true[k]) * sigmoid_derivative(self.z2[k])
            for k in range(n_out)
        ]

        # Hidden layer gradient  (W2ᵀ δ2 · sigmoid')
        delta1 = [
            sum(self.W2[k][j] * delta2[k] for k in range(n_out))
            * sigmoid_derivative(self.z1[j])
            for j in range(n_hid)
        ]

        # Update W2, b2
        for k in range(n_out):
            for j in range(n_hid):
                self.W2[k][j] -= self.lr * delta2[k] * self.a1[j]
            self.b2[k] -= self.lr * delta2[k]

        # Update W1, b1
        for j in range(n_hid):
            for i in range(len(self.x)):
                self.W1[j][i] -= self.lr * delta1[j] * self.x[i]
            self.b1[j] -= self.lr * delta1[j]

    # ── training ──

    def train(self, X, Y, epochs=1000, verbose=True):
        for epoch in range(1, epochs + 1):
            preds = []
            for x, y in zip(X, Y):
                out = self.forward(x)
                self.backward(y)
                preds.append(out)

            if verbose and epoch % 200 == 0:
                loss = binary_cross_entropy(
                    [y[0] for y in Y], [p[0] for p in preds]
                )
                print(f"  Epoch {epoch:5d} | Loss: {loss:.4f}")

    def predict(self, X):
        return [self.forward(x) for x in X]


# ── Demo — XOR problem ────────────────────────────────────────────────────────
if __name__ == "__main__":
    # XOR dataset
    X = [[0, 0], [0, 1], [1, 0], [1, 1]]
    Y = [[0],    [1],    [1],    [0]]

    net = MLP(n_input=2, n_hidden=4, n_output=1, lr=0.5, seed=7)

    print("=== Backpropagation — XOR ===")
    net.train(X, Y, epochs=1000)

    print("\n  Predictions after training:")
    for x, y in zip(X, Y):
        pred = net.forward(x)[0]
        label = 1 if pred >= 0.5 else 0
        print(f"  Input: {x}  →  Raw: {pred:.4f}  →  Class: {label}  (Target: {y[0]})")
    print()

=== Backpropagation — XOR ===
  Epoch   200 | Loss: 0.6951
  Epoch   400 | Loss: 0.5966
  Epoch   600 | Loss: 0.4414
  Epoch   800 | Loss: 0.2405
  Epoch  1000 | Loss: 0.1504

  Predictions after training:
  Input: [0, 0]  →  Raw: 0.0968  →  Class: 0  (Target: 0)
  Input: [0, 1]  →  Raw: 0.8379  →  Class: 1  (Target: 1)
  Input: [1, 0]  →  Raw: 0.8738  →  Class: 1  (Target: 1)
  Input: [1, 1]  →  Raw: 0.1660  →  Class: 0  (Target: 0)

